<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
DeepAgents: Parameter, Sandbox & Einordnung
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Der PMO-Assistent für Projekt Kompass nutzt zusätzliche Harness-Parameter für quellengebundene Briefings und menschliche Kontrolle von Dateiaktionen. Die optionale Sandbox-Referenz zeigt eine isolierte Auswertung von PMO-Zahlen.

**Empfohlener Lernpfad:** [Harness-Grundlagen](M50_DeepAgents_Harness.ipynb) → [PMO-Skill](M51_DeepAgents_Skill_PMO_Assistent.ipynb) → [mehrere PMO-Skills](M52_DeepAgent_Multi_Skill.ipynb) → [optionale Parameter- und Sandbox-Vertiefung](M53_DeepAgents_Vertiefung.ipynb). Der Lernpfad führt vom Grundaufbau über einen wiederverwendbaren Arbeitsstandard zur Kombination mehrerer Skills. Die Parameter- und Sandbox-Vertiefung ergänzt diesen Ablauf bei Bedarf.


**Praxis-Transfer:** Mara Vogt möchte einen PMO-Ergebnisentwurf erst nach Prüfung im virtuellen Agenten-Dateisystem speichern lassen. Der PMO-Assistent unterbricht vor der Dateiaktion; sie kann genehmigen oder ablehnen. Die optionale Sandbox-Vertiefung behandelt zusätzlich begrenzte Berechnungen, deren Ergebnis keine Budgetfreigabe erteilt.


In [ ]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul
!uv pip install --system -q deepagents==0.7.34

import os
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGSMITH_PROJECT"]  = "m53_deepagents_vertiefung"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace,
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import (
    BASELINE,
    ROUTER,
    WORKER,
    CODING,
    JUDGE,
    PLANNER,
    WORKER_PREMIUM,
    FRONTIER,
    IMAGE_GENERATION,
    TRANSCRIPTION,
    TRANSCRIPTION_SEGMENTS,
    EMBEDDINGS,
)
# LangSmith Tracing
run_cfg = {
    "tags": ["deepagents"],
    "metadata": {"notebook": "deepagents_vertiefung", "version": "1.0"}
}

In [ ]:
#@title 📂 Google Drive mounten (PMO-RAG) { display-mode: "form" }
from google.colab import drive

drive.mount("/content/drive")

# 1 | Übersicht
---

Dieses optionale Notebook vertieft den PMO-Koordinator aus den Harness-Grundlagen: `decision_analyst` recherchiert Beschlüsse und offene Zugriffsregeln, `risk_analyst` prüft belegte Risikostände. Beide nutzen den gemeinsamen PMO-RAG-Index aus den RAG-Grundlagen beziehungsweise dem RAG-Agenten. Das Notebook ist eigenständig ausführbar; die anderen DeepAgents-Notebooks müssen nicht zuvor in derselben Runtime laufen.


In [ ]:
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langsmith import tracing_context
from IPython.display import Image, display

**PMO-RAG als Datenquelle**

Die folgenden Tools nutzen die vorhandene Chroma-Collection aus M11/M12. In Colab zuerst Google Drive verbinden; der Index wird hier nur gelesen. Bei einer leeren oder veralteten Collection M11/M12 erneut zur Indexierung ausführen.

Die Tools liefern Fundstellen, keine vorformulierte Antwort: **abrufen → formatieren → durch den Agenten auswerten**. Der Retriever filtert auf `projekt_kompass` und freigegebene Quellen. Alte freigegebene Beschlüsse bleiben für den zeitlichen Vergleich verfügbar; Arbeitsnotizen, überholte Entwürfe und Fremdprojekte werden ausgeschlossen. „Freigegebene Quelle“ bedeutet keine Freigabe eines neuen Briefings oder produktiver Kundendaten.

Die Auswertung muss Quellen und Dokumentdatum nennen, D1 von D2 sowie Testkorpus von Produktivdaten unterscheiden. Fehlt ein Beleg, lautet die Antwort „Nicht im Korpus – in den abgerufenen Fundstellen nicht belegt“. Eine begrenzte Suche beweist keine vollständige Abwesenheit im Korpus. Vektorsuche allein garantiert keine inhaltliche Relevanz oder vollständige Evidenz.


In [ ]:
from genai_lib.pmo_rag import get_pmo_vectorstore, format_docs

# Bestehenden Index aus M11/M12 öffnen; hier keine Neuindexierung durchführen.
vectorstore = get_pmo_vectorstore()
if not vectorstore.get(limit=1)["ids"]:
    raise RuntimeError("PMO-Collection ist leer. Zuerst M11/M12 zur Indexierung ausführen.")

pmo_filter = {"$and": [
    {"project_id": {"$eq": "projekt_kompass"}},
    {"source_status": {"$in": ["approved_record", "approved_current"]}},
]}
if not vectorstore.get(where=pmo_filter, limit=1)["ids"]:
    raise RuntimeError("Keine freigegebenen Kompass-Quellen mit Metadaten. Index in M11/M12 aktualisieren.")
pmo_retriever = vectorstore.as_retriever(search_kwargs={"k": 8, "filter": pmo_filter})
print("Bestehende PMO-Collection geöffnet; Projekt- und Quellenfilter aktiv.")


In [ ]:
def retrieve_pmo_context(question: str, focus: str) -> str:
    """Ruft freigegebene Kompass-Fundstellen ab und formatiert sie ohne Modellaufruf."""
    # 1. Expliziter Abruf aus der bestehenden Wissensdatenbank.
    documents = pmo_retriever.invoke(f"Projekt Kompass: {question} Schwerpunkt: {focus}")
    if not documents:
        return "Nicht im Korpus – in den abgerufenen Fundstellen nicht belegt. Die Suche ist begrenzt."
    # 2. Quellen und Metadaten sichtbar formatieren; der Agent übernimmt die Auswertung.
    return format_docs(documents)

@tool
def search_project_decisions(question: str) -> str:
    """Sucht Belege zu Kompass-Entscheidungen, D1/D2, Vektordatenbank und offenen Zugriffsregeln."""
    return retrieve_pmo_context(question, "Beschlussstand, Datum, Prototyp versus Produktivbetrieb, offene Entscheidungen")

@tool
def search_project_risks(question: str) -> str:
    """Sucht Belege zu Kompass-Risiken und Datenschutz; liefert keine Freigabeentscheidung."""
    return retrieve_pmo_context(question, "R2, Datenschutzfreigabe, Testkorpus versus produktive Kundendaten")


In [ ]:
PMO_RULES = (
    "Verwende nur belegte Projekt-Kompass-Fakten. Nenne Quelle und Dokumentdatum. "
    "Unterscheide historische Beschlüsse von aktuellem Stand und Testkorpus von Produktivdaten. "
    "Fehlen Belege, antworte: Nicht im Korpus – in den abgerufenen Fundstellen nicht belegt. Behaupte keine vollständige Abwesenheit im gesamten Korpus. Fehlende Termine und Zuständigkeiten bleiben offen. "
    "Erstelle einen Entwurf; keine eigenständige Freigabe, kein Versand. Antworte auf Deutsch."
)


In [ ]:
# Die beiden Sub-Agenten erhalten je ein fachlich begrenztes RAG-Tool.
sub_decision_analyst = {
    "name": "decision_analyst",
    "description": "Recherchiert Kompass-Beschlüsse, D1/D2 und offene Rechte- und Rollenentscheidungen",
    "system_prompt": "Nutze search_project_decisions. Liefere Befund, Quelle, Datum und offene Punkte. " + PMO_RULES,
    "tools": [search_project_decisions],
    "model": init_chat_model(WORKER),
}

sub_risk_analyst = {
    "name": "risk_analyst",
    "description": "Recherchiert Kompass-Risiken, R2 und den Geltungsbereich der Datenschutzfreigabe",
    "system_prompt": "Nutze search_project_risks. Liefere Befund, Quelle, Datum und Freigabegrenzen. " + PMO_RULES,
    "tools": [search_project_risks],
    "model": init_chat_model(WORKER),
}

# 2 | Optionale Parameter & Sandbox-Backends
---

<p><font color='black' size="5">
Optional: Weitere Parameter
</font></p>

`create_deep_agent()` bietet weitere praxisrelevante Parameter. Die folgenden Parameter sind Referenzmaterial; der Kern dieses Moduls bleibt Planning, Filesystem und Sub-Agenten:

| Parameter | Beschreibung |
|-----------|-------------|
| `interrupt_on` | HITL-Freigabe vor bestimmten Tool-Calls; auch für Filesystem-Permissions |
| `memory` | Kontext-Dateien (AGENTS.md) beim Agent-Start automatisch laden |
| `skills` | SKILL.md-Dateien für den Hauptagenten laden |
| `permissions` | Filesystem-Zugriffsrechte einschränken — `list[FilesystemPermission]` |
| `response_format` | Strukturierte Agent-Ausgabe — Pydantic-Modell oder `dict` |
| `name` | Agent-Name für LangSmith-Tracing |
| `cache` | Response-Caching — `BaseCache`-Instanz |
| `backend` | Filesystem-Backend — In-Memory, Local, LangGraph Store oder Sandbox |
| `store` | LangGraph Memory Store für thread-übergreifende Persistenz |
| `context_schema` | Pydantic-Schema für den geteilten Agenten-Kontext |
| `state_schema` | Eigenes State-Schema übergeben |
| `debug` | Debug-Ausgaben aktivieren (`True` für detailliertes Logging) |

> ✅ **Sub-Agent-Modell:** Jeder Sub-Agent kann ein eigenes Modell über den `model=`-Schlüssel
> im Dict erhalten. `create_deep_agent()` selbst hat keinen `subagent_model`-Parameter —
> das Modell wird pro Sub-Agenten-Dict gesetzt:
> ```python
> research_subagent = {
>     "name": "recherche",
>     "description": "Recherchiert belegte Kompass-Beschlüsse",
>     "system_prompt": "Nenne Quellen und Datum; keine Freigabe erteilen.",
>     "model": init_chat_model(WORKER),   # ← eigenes Modell
>     "tools": [search_project_decisions],
> }
> ```

> 📌 **Weitere Bausteine:**
> - `CodeInterpreterMiddleware` (experimentell, `deepagents[quickjs]`) — JavaScript in einer QuickJS-Sandbox ausführen
> - `DeepAgentState` ist direkt aus `deepagents` importierbar
> - `interrupt_on` greift auch für Filesystem-Permissions

In [ ]:
# subagents und tools aus DeepAgents Vertiefung weiterverwenden (siehe Setup-Zelle oben)

extended_agent = create_deep_agent(
    model=init_chat_model(PLANNER),
    tools=[search_project_decisions, search_project_risks],
    subagents=[sub_decision_analyst, sub_risk_analyst],
    # HITL — write_file und execute erfordern menschliche Freigabe
    # interrupt_on greift auch für Filesystem-Permissions
    interrupt_on={"write_file": True, "execute": True},
    # Kontext-Dateien beim Start laden (AGENTS.md-Konvention)
    # memory=["./AGENTS.md"],  # Kommentiert: Datei muss existieren
    system_prompt=(
        "Du koordinierst Kompass-Briefings für Mara Vogt. Delegiere Beschlüsse an decision_analyst "
        "und Risiken an risk_analyst. Übergib Projekt, Stichtag und Teilfrage explizit. "
        + PMO_RULES
    ),
    checkpointer=InMemorySaver(),
)

**Was passiert hier?**

1. `InMemorySaver()` — speichert den Graph-Zustand im RAM für Session-Persistenz
2. `create_deep_agent(...)` — erstellt einen DeepAgent mit erweiterter Reasoning-Fähigkeit

✅ **Extended Agent** erstellt mit:

| Parameter | Wert |
|-----------|------|
| `interrupt_on` | `write_file=True`, `execute=True` — HITL aktiv |
| `memory` | auskommentiert (Pfad muss existieren) |

In [ ]:
with tracing_context(enabled=True):
    pmo_result = extended_agent.invoke(
        {"messages": [HumanMessage(content=(
            "Erstelle für Mara Vogt einen Briefing-Entwurf zu Projekt Kompass, Stand 20.05.2026. "
            "Recherchiere D2, den Geltungsbereich der R2-Freigabe und offene Zugriffsregeln. "
            "Nenne Quellen und Dokumentdatum. Keine Datei schreiben und nichts ausführen oder versenden."
        ))]},
        config={"configurable": {"thread_id": "m53_pmo_briefing"},
                "recursion_limit": 80, "run_name": "m53_kap2_pmo_briefing"},
    )
mprint(pmo_result["messages"][-1].text)

**PMO-Kontrollpunkt:** `interrupt_on` schützt hier Dateiaktionen. Die Recherche-Demo schreibt keine Datei und löst daher keine Freigabe aus. Teste optional eine Aufforderung, den Entwurf mit `write_file` zu speichern: Der Lauf muss vor dem Schreiben unterbrechen. Erst nach menschlicher Prüfung darf mit derselben `thread_id` fortgesetzt werden. Eine genehmigte Dateiaktion ist keine fachliche Briefing- oder Datenschutzfreigabe.


<p><font color='black' size="5">
Optional: Sandbox Backends
</font></p>

Für die Kern-Demo sind keine Remote-Sandboxes nötig. DeepAgents kann Code in **isolierten Remote-Sandboxes** ausführen — ohne Zugriff auf lokale Dateien, Credentials oder das Host-System.

| Backend | Voraussetzung | Typischer Einsatz |
|---------|---------------|-------------------|
| **Modal** | `modal setup` (CLI) | Serverless Python, kostengünstig |
| **Daytona** | `DAYTONA_API_KEY` | Managed Dev-Environments |
| **Runloop** | `RUNLOOP_API_KEY` | Enterprise Sandboxes |
| **LangSmith Sandbox** | LangSmith Account | Integriertes Tracing |

**Empfohlenes Pattern: "Remote Tool"**

> Agent läuft lokal (API-Keys sicher), Code-Ausführung passiert im Sandbox.
> Kosten entstehen nur für die reine Execution-Zeit.

<p><font color='darkblue' size="4">🚨 <b>Achtung</b></font></p>

Für die Demos in diesem Notebook wird kein Sandbox benötigt. Der folgende Code ist auskommentiert und dient als Referenz-Template.

In [ ]:
# ⚠️ Auskommentiert: Modal/Daytona/Runloop Account + API-Key erforderlich
# Dieses Template zeigt die Struktur — zum Ausführen Account aktivieren
# Sandbox-Anbindungen sind eigene Pakete: langchain-modal, langchain-daytona, langchain-runloop

# ── Modal Backend ────────────────────────────────────────────────────────
# import modal
# from langchain_modal import ModalSandbox
#
# app = modal.App.lookup("pmo_budget_demo", create_if_missing=True)
# sb  = modal.Sandbox.create(app=app)
# backend = ModalSandbox(sandbox=sb)
#
# sandbox_agent = create_deep_agent(
#     model=init_chat_model(CODING),
#     backend=backend,   # ← Code läuft im Sandbox, nicht lokal
#     system_prompt="Prüfe die übergebenen synthetischen PMO-Zahlen per Python. Keine Projektfreigabe erteilen.",
# )
# result = sandbox_agent.invoke({"messages": [
#     {"role": "user", "content": "Projekt Kompass, Demo-Zahlen: Budget 120000 EUR, Forecast 124500 EUR. Berechne die Abweichung absolut und prozentual; zeige den Code. Die Rechnung ist keine Budgetfreigabe."}
# ]}, config=run_cfg)

In [ ]:
#@markdown   <p><font size="4" color='green'>  Architektur: Remote Tool Pattern</font> </br></p>

# ── Architektur: Remote Tool Pattern ─────────────────────────────────────
diagram = '''
%%{init: {
  'theme': 'base',
  'themeVariables': {
    'primaryColor': '#e0f2fe',
    'primaryBorderColor': '#0284c7',
    'primaryTextColor': '#1e293b',
    'lineColor': '#64748b',
    'clusterBkg': '#f8fafc',
    'clusterBorder': '#cbd5e1',
    'noteBkgColor': '#fef3c7',
    'noteBorderColor': '#d97706',
    'noteTextColor': '#78350f',
    'edgeLabelBackground': '#ffffff'
  }
}}%%
flowchart LR
    subgraph lokal ["Lokaler Agent (API-Keys sicher)"]
        A["create_deep_agent(\nbackend=sandbox)"]
    end
    subgraph sandbox ["Remote Sandbox (Modal / Daytona / Runloop)"]
        C["execute(code)"]
        D["ls / read_file\nwrite_file / edit_file"]
        E[("Isoliertes\nFilesystem")]
    end
    A -->|"Tool Call"| C
    A -->|"Tool Call"| D
    C --- E
    D --- E
    C -->|"stdout / stderr"| A
    D -->|"Datei-Inhalt"| A

    %% Farbdefinitionen nach Funktionsgruppe
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef target fill:#fae8ff,stroke:#c026d3,stroke-width:2px,color:#701a75;

    %% Zuweisung
    class E target;
'''
mermaid(diagram, width=650)
mprint("**Remote Tool Pattern:** Agent lokal · Code-Ausführung im Sandbox · API-Keys nie im Sandbox")

# 3 | Optionaler Deep Dive: Vergleich & Einordnung
---


<p><font color='black' size="5">
Wann LangGraph manuell, wann DeepAgents?
</font></p>

DeepAgents ist kein Ersatz für LangGraph — es ist eine Schicht darüber.
Wähle das richtige Werkzeug je nach Anforderung:

| Kriterium | LangGraph manuell | DeepAgents Harness |
|-----------|-------------------|--------------------|
| **Transparenz** | ✅ Vollständig (jeder Node nachvollziehbar) | ⚠️ Gekapselt (Harness entscheidet) |
| **Kontrolle** | ✅ Präzise (Custom Routing, Custom State) | ⚠️ Opinionated (Harness-Defaults) |
| **Code-Aufwand** | ⚠️ Höher (State, Nodes, Routing explizit) | ✅ Minimal (`create_deep_agent()`) |
| **Planning** | ⚠️ Selbst bauen | ✅ `write_todos` (per `TodoListMiddleware`, Opt-in) |
| **Filesystem** | ⚠️ Selbst bauen | ✅ `ls`, `read_file`, `write_file` automatisch |
| **Sub-Agenten** | ⚠️ Aufwändig | ✅ `subagents=[...]` Parameter |
| **Debugging** | ✅ Direkt (man sieht jeden Schritt) | ⚠️ Erfordert LangGraph-Kenntnisse |
| **API-Stabilität** | ✅ LangGraph 1.0 — produktionserprobt | ⚠️ DeepAgents 0.7.x — API-Änderungen zwischen Versionen möglich |
| **Use Case** | Custom Workflows, präzise Kontrolle | Autonome Long-Running Tasks |

**Faustregel:**
- ✅ LangGraph manuell: wenn jeder Schritt exakt kontrolliert werden muss
- ✅ DeepAgents: wenn Planning, Dateiverwaltung oder Delegation als Harness-Funktionen benötigt werden

<p><font color='darkblue' size="4">🚨 <b>Achtung</b></font></p>

DeepAgents setzt LangGraph-Grundkenntnisse (StateGraph bis Supervisor Pattern) voraus. Die Harness bricht — und dann braucht man das Fundament, um zu debuggen. Die LangGraph-Grundlagen sind keine Vorbereitung auf die Abkürzung, sondern die Voraussetzung dafür, sie sinnvoll einzusetzen.

---

**Zur API-Stabilität — eine sachliche Einschätzung:**

DeepAgents 0.7.x ist eine junge Bibliothek. Das `subagents=`-Interface wurde in diesem
Notebook bereits korrigiert: Die Schnittstelle erwartet ein **Dict**, nicht einen
`CompiledStateGraph` — ein Breaking Change, der in keiner offiziellen Migrationsanleitung
dokumentiert war.

Das ist kein Einzelfall für frühe 0.x-Releases. Konkrete Konsequenz für den produktiven Einsatz:

```python
**Empfehlung: Version pinnen — in diesem Notebook bereits gesetzt:**
pip install deepagents==0.7.34   # Setup-Zelle oben verwendet diese Version
```

**Auswahl für den Betrieb:** Die Entscheidung richtet sich nach den Anforderungen des PMO-Ablaufs. LangGraph manuell passt zu festen Prüfpfaden und präziser Kontrolle über Routing und State. DeepAgents passt, wenn Planning, Dateiverwaltung und Delegation als Harness-Funktionen benötigt werden. Für eine einfache Tool-Aufgabe kann ein LangChain-Agent mit `create_agent()` genügen.

Der Produktionseinsatz führt nicht pauschal zu einer bestimmten Variante. Für jede Variante sind die gepinnten Paketversionen, Tests der konkreten Abläufe, erforderliche Freigaben und Monitoring zu prüfen. Ein Versionssprung allein ersetzt diese Prüfung nicht.


In [ ]:
#@markdown Entscheidungsbaum: Anforderungen des PMO-Ablaufs

diagram = '''
flowchart TD
    request(["PMO-Aufgabe"]) --> control{"Fester Ablauf und<br/>präzise Kontrolle nötig?"}
    control -->|Ja| manual["LangGraph manuell"]
    control -->|Nein| harness{"Planning, Dateien oder<br/>Delegation benötigt?"}
    harness -->|Ja| deep_agent["DeepAgents"]
    harness -->|Nein| simple_agent["Einfacher LangChain-Agent"]
    manual --> operations["Für den Betrieb prüfen:<br/>Tests, Freigaben, Monitoring"]
    deep_agent --> operations
    simple_agent --> operations
    classDef default fill:#e0f2fe,stroke:#0284c7,stroke-width:2px,color:#0c4a6e;
    classDef input fill:#e0e7ff,stroke:#6366f1,stroke-width:2px,color:#1e1b4b;
    classDef metric fill:#fef3c7,stroke:#d97706,stroke-width:2px,color:#78350f;
    classDef output fill:#dcfce7,stroke:#16a34a,stroke-width:2px,color:#14532d;
    class request input;
    class control,harness metric;
    class manual,deep_agent,simple_agent output;
'''
mermaid(diagram, width=750)


<p><font color='black' size="5">
Optionaler Kontext: Warum jetzt?
</font></p>

2024/2025 sind autonome Agenten-Systeme mit Harness-Pattern viral gegangen.
LangChain hat in einem Blog-Post die gemeinsame Architektur analysiert:

| System | Entwickler | Planning | Filesystem | Sub-Agenten |
|--------|-----------|----------|------------|-------------|
| **Claude Code** | Anthropic | ✅ Todo-Liste | ✅ read/write/edit | ✅ Sub-Tasks |
| **Manus** | Monica | ✅ Planung | ✅ Dateiverwaltung | ✅ Delegation |
| **Deep Research** | OpenAI | ✅ Search-Plan | ✅ Context-Offloading | ⚠️ Begrenzt |
| **DeepAgents** | LangChain | ✅ `write_todos` | ✅ ls/read/write | ✅ `subagents=[]` |

> *"Applications like Deep Research, Manus, and Claude Code have gotten around the*
> *limitation of shallow agents by implementing a combination of four things:*
> *a planning tool, sub agents, access to a file system, and a detailed prompt."*
> — LangChain Blog

# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


**Grundlagen**     
Vergleiche DeepAgents und LangGraph mit derselben PMO-Aufgabe: Fasse die drei unten angegebenen Kompass-Notizen in fünf Stichpunkten für Mara Vogt zusammen. Nutze nur diese Eingaben, keine zusätzliche RAG-Recherche. Quelle und Datum bleiben erhalten; unbelegte Angaben werden als „Nicht im Korpus“ markiert.

Baue für den Vergleich eine eigene DeepAgents-Instanz ohne Recherche-Tools und ohne zusätzliche Sub-Agenten sowie einen kleinen manuellen LangGraph. Verwende dafür nicht unverändert `extended_agent`, dessen Koordinator-Prompt die Delegation verlangt. Verwende dasselbe Modellprofil und dieselbe Eingabe. Dokumentiere Codezeilen ohne Setup/Importe, die Medianlatenz aus je drei Läufen sowie fünf Stichpunkte, Belegmarkierung und keine erfundenen Fakten als Qualitätskriterien. Eine Vorlage ohne Läufe ist kein Messergebnis.


In [ ]:
# Grundlagen: DeepAgents vs. LangGraph
import statistics
import time

task = """
Erstelle ein internes Kompass-Briefing für Mara Vogt mit fünf Stichpunkten.
Nutze nur diese Notizen; keine zusätzliche Recherche, keine Freigabe.
Quelle für alle Notizen: statusbericht_projekt_kompass_2026-05-20.md, Stand 20.05.2026.
1. D2 vom 14.04.2026: verwaltete Cloud-Vektordatenbank für Produktion; Anbieter offen.
2. R2 seit 28.04.2026 mitigiert für den Testkorpus; keine Freigabe produktiver Kundendaten.
3. Rechte- und Rollenentscheidung am 19.05.2026 vertagt; Mara Vogt bereitet eine Vorlage vor.
Markiere fehlende Evidenz als Nicht im Korpus und erhalte Quelle und Datum.
"""

# Dieselbe Eingabe für beide Varianten verwenden.
# 1. DeepAgents: eigene Instanz ohne RAG-Tools und zusätzliche Sub-Agenten bauen.
#    Beide Varianten verwenden PLANNER und nur die übergebenen Notizen.
# 2. LangGraph: kleinen manuellen Graphen mit Analyse- und Schreib-Node bauen.
# 3. Je Variante drei Latenzen messen und den Median berechnen.
# 4. LOC ohne Setup-/Importzellen eintragen.
# 5. Qualität mit drei Kriterien bewerten: fünf Stichpunkte, Belegmarkierung,
#    keine erfundenen Fakten.
comparison = [
    ["DeepAgents", None, None, None],
    ["LangGraph", None, None, None],
]
print(task)
print("Noch nicht gemessen (Codezeilen, Medianlatenz, Qualität):", comparison)


**Aufbau — Briefing-Datei erst nach menschlicher Prüfung speichern**

Erstelle eine eigene DeepAgents-Instanz für die Ablage eines bereits formulierten Kompass-Briefing-Entwurfs. Sie benötigt keine RAG-Tools oder zusätzlichen Sub-Agenten. Verwende `InMemorySaver()` und `interrupt_on={"write_file": True}`. Ziel ist `/briefings/kompass_draft.md` im virtuellen Agenten-Dateisystem.

Fordere den Agenten auf, den unten vorgegebenen Text unverändert zu speichern. Prüfe zuerst die vorgeschlagene Dateiaktion und ihren Inhalt. Vor der Entscheidung muss der Lauf unterbrochen sein; die Datei darf noch nicht im State stehen.

Teste mit zwei getrennten Sessions:

- **Genehmigen:** Nach Prüfung mit `Command(resume={"decisions": [{"type": "approve"}]})` und derselben `thread_id` fortsetzen. Danach müssen Dateipfad und Inhalt im State nachweisbar sein.
- **Ablehnen:** In einer neuen Session dieselbe Aktion mit `type="reject"` ablehnen. Die Datei darf nicht angelegt werden; weitere Schreibversuche ebenfalls nicht genehmigen.

Nutze `get_state(config).values` zur Kontrolle des virtuellen Dateisystems. Die Reihenfolge und Anzahl der Entscheidungen muss zu den angezeigten `action_requests` passen. Das ist eine Freigabe der Dateiaktion, keine fachliche Briefing-, Datenschutz- oder Versandfreigabe. In Colab entsteht dadurch keine Datei im Betriebssystem.


In [ ]:
# Aufbau: Startgerüst; Agent und Prüfungen selbst ergänzen.
from langgraph.types import Command

draft_path = "/briefings/kompass_draft.md"
draft_text = (
    "Projekt Kompass – Entwurf, Stand 20.05.2026. "
    "D2: Cloud-Vektordatenbank für Produktion vorgesehen; Anbieter offen. "
    "R2-Freigabe nur für den Testkorpus. Quelle: statusbericht_projekt_kompass_2026-05-20.md. "
    "Menschliche Briefing-Freigabe offen; kein Versand."
)

# 1. approval_agent mit PLANNER, InMemorySaver und interrupt_on bauen.
# 2. Zwei neue thread_ids für Genehmigung und Ablehnung festlegen.
# 3. Speicheranfrage ausführen und result["__interrupt__"] prüfen.
# 4. action_requests ansehen; vor der Entscheidung darf draft_path nicht in files stehen.
# 5. Nach menschlicher Prüfung mit passendem Command und derselben config fortsetzen.
# 6. final_state = approval_agent.get_state(config).values
# 7. Bei Genehmigung: final_state["files"][draft_path]["content"] == draft_text prüfen.
#    Bei Ablehnung muss draft_path im virtuellen Dateisystem fehlen.


**Vertiefung — Budgetauswertung mit begrenzter Sandbox-Ausführung**

Erweitere das Remote-Tool-Muster um eine kontrollierte PMO-Rechnung. Verwende die unten stehenden synthetischen Testfälle; die Zahlen sind keine neue Projektentscheidung. Das Modell darf fehlende Werte nicht ergänzen.

Trenne drei Schritte: Eingaben validieren, einen festen Berechnungsauftrag ausführen und das Ergebnis mit Eingabequelle sowie Status „Entwurf“ ausgeben. Der Rechenkern soll absolute Abweichung (`forecast - budget`) und relative Abweichung in Prozent bestimmen. Bei Budget `0` ist die relative Abweichung nicht definiert; dieser Fall wird nicht als `0 %` ausgegeben. Bei fehlendem Forecast muss der Ablauf vor der Ausführung stoppen.

Teste:

- Budget 120.000 EUR und Forecast 124.500 EUR: **+4.500 EUR, +3,75 %**.
- Budget 120.000 EUR und Forecast 118.000 EUR: **−2.000 EUR, ungefähr −1,67 %**.
- Budget 0 EUR: relative Abweichung **nicht definiert**.
- Fehlender Forecast: **Rückfrage**, kein Ausführungsaufruf.

Eine echte Remote-Sandbox ist optional und benötigt den jeweiligen Account. Ohne Account verwende einen ausdrücklich als simuliert gekennzeichneten Executor für den festen Rechenkern und protokolliere dessen Aufrufe. Simulierte Tests belegen keine Remote-Isolation. Vermeide die Ausführung frei generierten Codes auf der Colab-VM. Weise Quelle, Rechenweg, Ausführungsart und Fehlerstatus aus; das Rechenergebnis erteilt keine Budgetfreigabe.


In [ ]:
# Vertiefung: feste Testfälle und Schnittstelle, keine Remote-Ressource wird gestartet.
budget_cases = [
    {"name": "over_budget", "budget": 120000, "forecast": 124500},
    {"name": "under_budget", "budget": 120000, "forecast": 118000},
    {"name": "zero_budget", "budget": 0, "forecast": 4500},
    {"name": "missing_forecast", "budget": 120000, "forecast": None},
]
execution_log = []

# 1. Eingabeprüfung: fehlende/nicht numerische Werte vor dem Executor erkennen.
# 2. Rechenkern: absolute Abweichung und Prozentsatz; bei Budget 0 Prozentwert None.
# 3. simulate_budget_execution(payload) oder einen bewusst aktivierten Remote-Executor bauen.
# 4. Nur den festen Rechenkern mit validierten Eingaben ausführen, keine freien Code-Strings.
# 5. Rückgabewerte und execution_log gegen die vier Erwartungen prüfen.
# 6. Bericht: Quelle "synthetischer M53-Testfall", Ausführungsart und Status "Entwurf".


<p><font color='darkblue' size="4">
 <b>✨ Viz</b>
</font></p>

- [LangGraph](https://ralf-42.github.io/Agenten/visualisierungen/langgraph_explainer/)
- [KI-Agenten-Architektur](https://ralf-42.github.io/Agenten/visualisierungen/ki_agent_architektur/)
- [Modellsteuerung](https://ralf-42.github.io/Agenten/visualisierungen/modellsteuerung_optimierung/)


# B | Dokumente zum Weiterlesen
---



Ergänzende Artikel aus der Kurs-Dokumentation:

- [LangGraph - Überblick](https://ralf-42.github.io/Agenten/05-frameworks/langgraph.html)
- [Checkliste Agentensystem](https://ralf-42.github.io/Agenten/04-agenten-implementierung/checkliste-agentensystem.html)
